# plot performance bar charts for paper


In [1]:
#imports libraries
import re,os,sys,math
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import re
import glob
import numpy as np
import natsort

In [2]:
title_shaheen='SB versus MWD modeling phase on the dual-socket 16-core Intel Haswell System @2.3GHz.'
title_kanary='SB versus MWD modeling phase on the dual-socket 128-core AMD Epyc Rome System @2.0GHz.'

In [3]:
# folders with logs
list_of_folders=['./tb_param_search/Shaheen']

grids_unique= ['512_512_512','1024_1024_512','2048_2048_512','2048_2048_1024']
grids_unique=['512_512_512','1024_1024_512','2048_2048_512']
# grids_unique= ['512_512_512','1024_1024_512']

## Aggregate all results

In [4]:
print('!!!!!!!!!!!!!!!!!!!!!!!!!1st order!!!!!!!!!!!!!!!!!!!!!!!!!')
type_of_code=r'-TB_1st_'
dataframes_list=[]
for folder in list_of_folders:
    path=os.path.join(folder,'*.log')
    FilenamesList = glob.glob(path)
    filenames_list=[]
    # wildcard_pattern = r'1st'
    wildcard_pattern = type_of_code
    for word in FilenamesList:
        match_word = re.search(wildcard_pattern, word)
        if match_word:
            filenames_list.append(word)
            # print(word)# Initialiser les listes pour stocker les données
    times=[]
    names=[]
    cases=[]
    giga_points=[]
    giga_flops = []
    point_updates = []
    flops=[]
    th_x=[]
    th_y=[]
    th_z=[]
    num_th=[]
    numwf=[]
    t_dim=[]
    grids=[]
    for filename in filenames_list:
    # for filename in filenames_list[1:3]:
        with open(filename,'r') as file:
            tmp_time=math.nan
            tmp_giga_points=math.nan
            tmp_giga_flops=math.nan
            tmp_point_updates=math.nan
            tmp_flops=math.nan
            for line in file:
                if 'ELAPSED TIME (s)' in line and np.isnan(tmp_time):
                    tmp_time=(float(line.split()[-1]))
                elif 'GIGA POINT / s' in line and np.isnan(tmp_giga_points):
                    tmp_giga_points=(float(line.split()[-1]))
                elif 'GIGA FLOP / s' in line and np.isnan(tmp_giga_flops):
                    tmp_giga_flops=(float(line.split()[-1]))
                elif '# POINT UPDATES' in line and np.isnan(tmp_point_updates):
                    tmp_point_updates=(float(line.split()[-1]))
                elif 'FLOP' in line and np.isnan(tmp_flops):
                    tmp_flops=(float(line.split()[-1]))   
        if np.isnan(tmp_time):
            continue
        else:
            times.append(tmp_time)
            giga_points.append(tmp_giga_points)
            giga_flops.append(tmp_giga_flops)
            point_updates.append(tmp_point_updates)
            flops.append(tmp_flops)
            tmp=filename.split('/')[-1]
            tmp=tmp.split('.log')[0]
            # print(tmp)
            # print(tmp.split('_'))
            num_th.append( float (tmp.split('_')[2]) )
            th_x.append(float(tmp.split('_')[3]))
            th_y.append(float(tmp.split('_')[4]))
            th_z.append(float(tmp.split('_')[5]))
            numwf.append(float(tmp.split('_')[6]))
            t_dim.append(float(tmp.split('_')[7]))
            grids.append( tmp.split('_')[8]+'_'+tmp.split('_')[9]+'_'+tmp.split('_')[10] )
            
            names.append(filename)
    # Créer un DataFrame à partir des listes
    data = pd.DataFrame({
        'folder':folder.split('/')[2],
        'names':names,
        'grids':grids,
        'num_th':num_th,
        'th_x':th_x,
        'th_y':th_y,
        'th_z':th_z,
        'numwf':numwf,
        't_dim':t_dim,
        'giga_point_s':giga_points,
        'giga_flop_s': giga_flops,
        'point_updates': point_updates,
        'Flops': flops,
        'times':times,
        })
    dataframes_list.append(data)
    # data.sort_values(by=['giga_point_s'],ascending=False)
    # print(data['giga_point_s'])
    
    num_threads=np.unique((data['num_th'])) 
    for num_th in num_threads:
        # print(num_th)
        data_f = data[data['num_th']==num_th]
        data_f=data_f.sort_values(by=['giga_point_s'],ascending=False)
        # print(data_f.iloc[0])
        
    # data.to_excel('./xls/tb_search_'+folder.split('/')[-1]+'.xlsx')
    # data.to_excel('./tb_search_'+folder.split('/')[-1]+'.xlsx')
# print(len(dataframes_list))

grid_list=[]
th_x_arr=[]
th_y_arr=[]
th_z_arr=[]
num_wf_arr=[]
tdim_arr=[]
gstencils=[]

# grids_unique=np.unique((data['grids'])) 
for grid in grids_unique:
    # print(grid)
    data_f = data[data['grids']==grid]
    data_f=data_f.sort_values(by=['giga_point_s'],ascending=False)
    # print(data_f.iloc[0])
    # print( data_f.iloc[0]['th_x'] )
    ######## aggregate results
    grid_list.append(grid)
    th_x_arr.append(data_f.iloc[0]['th_x'])
    th_y_arr.append(data_f.iloc[0]['th_y'])
    th_z_arr.append(data_f.iloc[0]['th_z'])
    num_wf_arr.append(data_f.iloc[0]['numwf'])
    tdim_arr.append(data_f.iloc[0]['t_dim'])
    gstencils.append(data_f.iloc[0]['giga_point_s'])
    print(data_f.iloc[0]['names'])

print('!!!!!!!!!!!!!!!!!!!!!!!!!1st order!!!!!!!!!!!!!!!!!!!!!!!!!')
print('grid_list=',grids_unique)
print('th_x_arr=',th_x_arr)
print('th_y_arr=',th_y_arr)
print('th_z_arr=',th_z_arr)
print('num_wf_arr=',num_wf_arr)
print('tdim_arr=',tdim_arr)
print('gstencils=',gstencils)

print('!!!!!!!!!!!!!!!!!!!!!!!!!2nd order!!!!!!!!!!!!!!!!!!!!!!!!!')
type_of_code=r'-TB_2nd_'
dataframes_list=[]
for folder in list_of_folders:
    path=os.path.join(folder,'*.log')
    FilenamesList = glob.glob(path)
    filenames_list=[]
    # wildcard_pattern = r'1st'
    wildcard_pattern = type_of_code
    for word in FilenamesList:
        match_word = re.search(wildcard_pattern, word)
        if match_word:
            filenames_list.append(word)
            # print(word)# Initialiser les listes pour stocker les données
    times=[]
    names=[]
    cases=[]
    giga_points=[]
    giga_flops = []
    point_updates = []
    flops=[]
    th_x=[]
    th_y=[]
    th_z=[]
    num_th=[]
    numwf=[]
    t_dim=[]
    grids=[]
    for filename in filenames_list:
    # for filename in filenames_list[1:3]:
        with open(filename,'r') as file:
            tmp_time=math.nan
            tmp_giga_points=math.nan
            tmp_giga_flops=math.nan
            tmp_point_updates=math.nan
            tmp_flops=math.nan
            for line in file:
                if 'ELAPSED TIME (s)' in line and np.isnan(tmp_time):
                    tmp_time=(float(line.split()[-1]))
                elif 'GIGA POINT / s' in line and np.isnan(tmp_giga_points):
                    tmp_giga_points=(float(line.split()[-1]))
                elif 'GIGA FLOP / s' in line and np.isnan(tmp_giga_flops):
                    tmp_giga_flops=(float(line.split()[-1]))
                elif '# POINT UPDATES' in line and np.isnan(tmp_point_updates):
                    tmp_point_updates=(float(line.split()[-1]))
                elif 'FLOP' in line and np.isnan(tmp_flops):
                    tmp_flops=(float(line.split()[-1]))   
        if np.isnan(tmp_time):
            continue
        else:
            times.append(tmp_time)
            giga_points.append(tmp_giga_points)
            giga_flops.append(tmp_giga_flops)
            point_updates.append(tmp_point_updates)
            flops.append(tmp_flops)
            tmp=filename.split('/')[-1]
            tmp=tmp.split('.log')[0]
            # print(tmp)
            # print(tmp.split('_'))
            num_th.append( float (tmp.split('_')[2]) )
            th_x.append(float(tmp.split('_')[3]))
            th_y.append(float(tmp.split('_')[4]))
            th_z.append(float(tmp.split('_')[5]))
            numwf.append(float(tmp.split('_')[6]))
            t_dim.append(float(tmp.split('_')[7]))
            grids.append( tmp.split('_')[8]+'_'+tmp.split('_')[9]+'_'+tmp.split('_')[10] )
            
            names.append(filename)
    # Créer un DataFrame à partir des listes
    data = pd.DataFrame({
        'folder':folder.split('/')[2],
        'names':names,
        'grids':grids,
        'num_th':num_th,
        'th_x':th_x,
        'th_y':th_y,
        'th_z':th_z,
        'numwf':numwf,
        't_dim':t_dim,
        'giga_point_s':giga_points,
        'giga_flop_s': giga_flops,
        'point_updates': point_updates,
        'Flops': flops,
        'times':times,
        })
    dataframes_list.append(data)
    # data.sort_values(by=['giga_point_s'],ascending=False)
    # print(data['giga_point_s'])
    
    num_threads=np.unique((data['num_th'])) 
    for num_th in num_threads:
        # print(num_th)
        data_f = data[data['num_th']==num_th]
        data_f=data_f.sort_values(by=['giga_point_s'],ascending=False)
        # print(data_f.iloc[0])
        
    # data.to_excel('./xls/tb_search_'+folder.split('/')[-1]+'.xlsx')
    # data.to_excel('./tb_search_'+folder.split('/')[-1]+'.xlsx')
# print(len(dataframes_list))

grid_list=[]
th_x_arr=[]
th_y_arr=[]
th_z_arr=[]
num_wf_arr=[]
tdim_arr=[]
gstencils=[]

# grids_unique=np.unique((data['grids'])) 
for grid in grids_unique:
    # print(grid)
    data_f = data[data['grids']==grid]
    data_f=data_f.sort_values(by=['giga_point_s'],ascending=False)
    # print(data_f.iloc[0])
    # print( data_f.iloc[0]['th_x'] )
    ######## aggregate results
    grid_list.append(grid)
    th_x_arr.append(data_f.iloc[0]['th_x'])
    th_y_arr.append(data_f.iloc[0]['th_y'])
    th_z_arr.append(data_f.iloc[0]['th_z'])
    num_wf_arr.append(data_f.iloc[0]['numwf'])
    tdim_arr.append(data_f.iloc[0]['t_dim'])
    gstencils.append(data_f.iloc[0]['giga_point_s'])
    print(data_f.iloc[0]['names'])

print('!!!!!!!!!!!!!!!!!!!!!!!!!2nd order!!!!!!!!!!!!!!!!!!!!!!!!!')
print('grid_list=',grids_unique)
print('th_x_arr=',th_x_arr)
print('th_y_arr=',th_y_arr)
print('th_z_arr=',th_z_arr)
print('num_wf_arr=',num_wf_arr)
print('tdim_arr=',tdim_arr)
print('gstencils=',gstencils)
print('!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!')
print('th_x unique=',np.unique((data['th_x'])))
print('th_y unique=',np.unique((data['th_y'])))
print('th_z unique=',np.unique((data['th_z'])))
print('numwf unique=',np.unique((data['numwf'])))
print('t_dim unique=',np.unique((data['t_dim'])))
print('gstencils unique=',np.unique((data['giga_point_s'])))

!!!!!!!!!!!!!!!!!!!!!!!!!1st order!!!!!!!!!!!!!!!!!!!!!!!!!
./tb_param_search/Flamingo/log-TB_1st_56_4_2_1_4_3_512_512_512.log
./tb_param_search/Flamingo/log-TB_1st_56_2_2_2_2_3_1024_1024_512.log
./tb_param_search/Flamingo/log-TB_1st_56_2_2_2_2_3_2048_2048_512.log
!!!!!!!!!!!!!!!!!!!!!!!!!1st order!!!!!!!!!!!!!!!!!!!!!!!!!
grid_list= ['512_512_512', '1024_1024_512', '2048_2048_512']
th_x_arr= [4.0, 2.0, 2.0]
th_y_arr= [2.0, 2.0, 2.0]
th_z_arr= [1.0, 2.0, 2.0]
num_wf_arr= [4.0, 2.0, 2.0]
tdim_arr= [3.0, 3.0, 3.0]
gstencils= [8.28796, 7.68736, 7.42707]
!!!!!!!!!!!!!!!!!!!!!!!!!2nd order!!!!!!!!!!!!!!!!!!!!!!!!!
./tb_param_search/Flamingo/log-TB_2nd_56_4_2_1_24_3_512_512_512.log
./tb_param_search/Flamingo/log-TB_2nd_56_2_2_1_16_3_1024_1024_512.log
./tb_param_search/Flamingo/log-TB_2nd_56_2_2_1_16_3_2048_2048_512.log
!!!!!!!!!!!!!!!!!!!!!!!!!2nd order!!!!!!!!!!!!!!!!!!!!!!!!!
grid_list= ['512_512_512', '1024_1024_512', '2048_2048_512']
th_x_arr= [4.0, 2.0, 2.0]
th_y_arr= [2.0, 2.0, 2.0]
th

In [5]:
grids_unique

['512_512_512', '1024_1024_512', '2048_2048_512']

## Filter all results according to number of threads

In [6]:
num_threads=np.unique((data['num_th'])) 
for num_th in num_threads:
    print(num_th)
    data_f = data[data['num_th']==num_th]
    data_f=data_f.sort_values(by=['giga_point_s'],ascending=False)
    print(data_f.iloc[0])

56.0
folder                                                    Flamingo
names            ./tb_param_search/Flamingo/log-TB_2nd_56_4_2_1...
grids                                                  512_512_512
num_th                                                        56.0
th_x                                                           4.0
th_y                                                           2.0
th_z                                                           1.0
numwf                                                         24.0
t_dim                                                          3.0
giga_point_s                                               13.2103
giga_flop_s                                                647.302
point_updates                                        37849399296.0
Flops                                              1854620565504.0
times                                                      2.86515
Name: 32, dtype: object


## Filter all results according to grid size, 1st order

In [10]:
data_f

,folder,names,grids,num_th,th_x,th_y,th_z,numwf,t_dim,giga_point_s,giga_flop_s,point_updates,Flops,times
429,Flamingo,./tb_param_search/Flamingo/log-TB_2nd_56_2_2_1...,2048_2048_512,56.0,2.0,2.0,1.0,16.0,3.0,11.97740,586.891,6.055904e+11,2.967393e+13,50.5612
243,Flamingo,./tb_param_search/Flamingo/log-TB_2nd_56_2_2_1...,2048_2048_512,56.0,2.0,2.0,1.0,4.0,3.0,11.85050,580.675,6.055904e+11,2.967393e+13,51.1025
334,Flamingo,./tb_param_search/Flamingo/log-TB_2nd_56_1_2_2...,2048_2048_512,56.0,1.0,2.0,2.0,2.0,3.0,11.68150,572.392,6.055904e+11,2.967393e+13,51.8420
416,Flamingo,./tb_param_search/Flamingo/log-TB_2nd_56_2_2_1...,2048_2048_512,56.0,2.0,2.0,1.0,2.0,3.0,11.49670,563.340,6.055904e+11,2.967393e+13,52.6750
58,Flamingo,./tb_param_search/Flamingo/log-TB_2nd_56_2_2_1...,2048_2048_512,56.0,2.0,2.0,1.0,20.0,3.0,11.49390,563.204,6.055904e+11,2.967393e+13,52.6878
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155,Flamingo,./tb_param_search/Flamingo/log-TB_2nd_56_2_1_2...,2048_2048_512,56.0,2.0,1.0,2.0,2.0,7.0,4.37132,214.195,6.227703e+11,3.051574e+13,142.4670
206,Flamingo,./tb_param_search/Flamingo/log-TB_2nd_56_1_2_2...,2048_2048_512,56.0,1.0,2.0,2.0,2.0,15.0,4.17518,204.584,6.227703e+11,3.051574e+13,149.1600
435,Flamingo,./tb_param_search/Flamingo/log-TB_2nd_56_1_1_2...,2048_2048_512,56.0,1.0,1.0,2.0,2.0,7.0,4.13538,202.634,6.227703e+11,3.051574e+13,150.5960
324,Flamingo,./tb_param_search/Flamingo/log-TB_2nd_56_2_1_2...,2048_2048_512,56.0,2.0,1.0,2.0,2.0,15.0,4.07756,199.801,6.227703e+11,3.051574e+13,152.7310


In [7]:
grid_list=[]
th_x_arr=[]
th_y_arr=[]
th_z_arr=[]
num_wf_arr=[]
tdim_arr=[]

grids_unique=np.unique((data['grids'])) 
for grid in grids_unique:
    # print(grid)
    data_f = data[data['grids']==grid]
    data_f=data_f.sort_values(by=['giga_point_s'],ascending=False)
    data_f = data[data['grids']==grid]
    # print(data_f.iloc[0])
    # print( data_f.iloc[0]['th_x'] )
    ######## aggregate results
    grid_list.append(grid)
    th_x_arr.append(data_f.iloc[0]['th_x'])
    th_y_arr.append(data_f.iloc[0]['th_y'])
    th_z_arr.append(data_f.iloc[0]['th_z'])
    num_wf_arr.append(data_f.iloc[0]['numwf'])
    tdim_arr.append(data_f.iloc[0]['t_dim'])

print('grid_list=',grid_list)
print('th_x_arr=',th_x_arr)
print('th_y_arr=',th_y_arr)
print('th_z_arr=',th_z_arr)
print('num_wf_arr=',num_wf_arr)
print('tdim_arr=',tdim_arr)

grid_list= ['1024_1024_512', '2048_2048_512', '512_512_512']
th_x_arr= [2.0, 2.0, 4.0]
th_y_arr= [2.0, 2.0, 2.0]
th_z_arr= [1.0, 1.0, 1.0]
num_wf_arr= [16.0, 16.0, 24.0]
tdim_arr= [3.0, 3.0, 3.0]


In [8]:
for grid in grids:
    # print(grid)
    data_f = data[data['grids']==grid]
    data_f=data_f.sort_values(by=['giga_point_s'],ascending=False)
    print(data_f.iloc[0])
    print(data_f.iloc[0]['names'])
    # print( data_f.iloc[0]['th_x'] )

folder                                                    Flamingo
names            ./tb_param_search/Flamingo/log-TB_2nd_56_2_2_1...
grids                                                2048_2048_512
num_th                                                        56.0
th_x                                                           2.0
th_y                                                           2.0
th_z                                                           1.0
numwf                                                         16.0
t_dim                                                          3.0
giga_point_s                                               11.9774
giga_flop_s                                                586.891
point_updates                                       605590388736.0
Flops                                             29673929048064.0
times                                                      50.5612
Name: 429, dtype: object
./tb_param_search/Flamingo/log-TB_2nd

In [9]:
# x=data.loc[data.names.str.contains('SB_1st_'),'elapsed_time'].values
# print(float(x))
dataframes_list[0].giga_point_s

0       8.25573
1       8.60968
2       9.39013
3      10.94230
4      10.52400
         ...   
470    10.87160
471     5.27577
472     6.01169
473     7.25320
474    11.26840
Name: giga_point_s, Length: 475, dtype: float64